<a href="https://colab.research.google.com/github/johanndeboda/AAI2026/blob/2026fall/ex1_prompt_chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 — Prompt Chaining for a Customer Support AI
**Tools:** Google Colab, Gemini API (google-genai SDK, gemini-3.8-flash, free tier)
**Scenario:** VoltCart, an online electronics store
**Chain:** Classify → Gather missing info → Propose solution → Escalation check
Each step returns JSON that is passed into the next step's prompt.


In [35]:
from google import genai
from google.colab import userdata
import json, time

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-2.5-flash"

def ask(prompt, as_json=False):
    """Send a prompt to Gemini. If as_json, parse the reply into a dict."""
    last_error = None
    for attempt in range(4):
        try:
            time.sleep(7)  # stay under the free-tier limit (~10 requests/min)
            text = client.interactions.create(model=MODEL, input=prompt).output_text.strip()
            if not as_json:
                return text
            return json.loads(text[text.find("{"): text.rfind("}") + 1])
        except json.JSONDecodeError as e:
            last_error = e
            prompt += "\n\nReturn ONLY valid JSON. No extra text."
        except Exception as e:
            last_error = e
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                time.sleep(30)
                continue
            raise
    raise RuntimeError(f"Failed after 4 attempts. Last error: {last_error}")

import textwrap

def show(title, data):
    print(f"\n=== {title} ===")
    text = json.dumps(data, indent=2) if isinstance(data, dict) else data
    for line in text.splitlines():
        print(textwrap.fill(line, width=110, subsequent_indent="    "))

In [26]:
POLICY = """
- Late delivery: if more than 5 business days late, offer free expedited reshipment or a full refund.
- Damaged item: free replacement or full refund. Customer must provide a photo.
- Billing: duplicate charges are refunded in 5-7 business days after verification.
- Refunds: allowed within 30 days of delivery for unused items.
- Never promise exact delivery dates. Never offer discounts or store credit outside this policy.
- Never ask for full card numbers or passwords.
"""

REQUIRED_INFO = {
    "late_delivery":  ["order_number", "shipping_zip_code"],
    "damaged_item":   ["order_number", "item_name", "photo_of_damage"],
    "billing":        ["order_number", "charge_date", "last_4_digits_of_card"],
    "refund_request": ["order_number", "delivery_date", "item_condition"],
    "technical":      ["product_model", "problem_description"],
    "other":          ["order_number"],
}

In [27]:
STEP1 = """You are a Tier-1 support triage agent for VoltCart, an online electronics store.
Classify the customer message below.

Rules:
- issue_type: exactly one of late_delivery, damaged_item, billing, refund_request, technical, other
- urgency: low, medium, or high (high = money lost, item unusable, or customer threatens a dispute)
- sentiment: calm, frustrated, or angry
- known_info: only details the customer actually stated (order number, item, amount). Do not guess.
- Return ONLY JSON in this format:
{{"issue_type": "", "urgency": "", "sentiment": "", "known_info": {{}}, "summary": ""}}

Customer message:
<<<{message}>>>"""

In [28]:
STEP2 = """You are a VoltCart support agent writing ONE follow-up message.

Case from Step 1:
{step1}

Details our team needs for a {issue_type} issue: {required}

Tasks:
1. Compare the needed details with known_info. List only what is still missing.
2. Write one follow-up message asking for all missing details at once.

Constraints:
- If sentiment is frustrated or angry, open with a one-sentence apology.
- Max 3 sentences. Do not propose a solution yet.
- Never ask for full card numbers or passwords.
- Return ONLY JSON: {{"missing_info": [], "follow_up_question": ""}}"""

In [29]:
STEP3 = """You are a senior VoltCart support agent resolving this case.

Case from Step 1:
{step1}

Customer's answer to our follow-up (Step 2):
<<<{answer}>>>

Company policy:
{policy}

Constraints:
- Only use actions allowed by the policy. If the policy does not cover the case, set proposed_action to "needs_review".
- customer_reply: 60-120 words, friendly and plain, uses the customer's details, ends with one clear next step.
- No exact delivery dates, discounts, or credits outside policy.
- Do not claim anything was verified or checked. Describe what happens next instead (e.g. "once we confirm...").
- Do not assume how many days late an order is unless the customer gave dates.
- Return ONLY JSON: {{"proposed_action": "", "policy_rule_used": "", "customer_reply": ""}}"""

In [30]:
STEP4 = """You are VoltCart's escalation checker. Decide if a human must review before the reply is sent.

Case from Step 1:
{step1}

Proposed resolution from Step 3:
{step3}

Escalate if ANY of these is true:
- sentiment is angry AND urgency is high
- proposed_action is "needs_review"
- customer mentions legal action, a chargeback/bank dispute, or posting on social media
- order value is over $500
Otherwise, do not escalate.

Return ONLY JSON: {{"escalate": true or false, "rule_triggered": "", "route_to": "tier2_human or none"}}"""

In [31]:
def run_chain(message, customer_answer):
    show("CUSTOMER MESSAGE", message)

    s1 = ask(STEP1.format(message=message), as_json=True)
    show("STEP 1 — CLASSIFY", s1)

    required = REQUIRED_INFO.get(s1["issue_type"], REQUIRED_INFO["other"])
    s2 = ask(STEP2.format(step1=json.dumps(s1), issue_type=s1["issue_type"],
                          required=required), as_json=True)
    show("STEP 2 — FOLLOW-UP QUESTION", s2)

    show("CUSTOMER ANSWER (simulated)", customer_answer)

    s3 = ask(STEP3.format(step1=json.dumps(s1), answer=customer_answer,
                          policy=POLICY), as_json=True)
    show("STEP 3 — PROPOSED SOLUTION", s3)

    s4 = ask(STEP4.format(step1=json.dumps(s1), step3=json.dumps(s3)), as_json=True)
    show("STEP 4 — ESCALATION", s4)

    final = "Sent to human agent for review." if s4["escalate"] else s3["customer_reply"]
    show("FINAL OUTCOME", final)

In [37]:
run_chain(
    "I just opened my new 65-inch TV from order VC-48213 and the screen is cracked. "
    "I paid $1,200 for this. If this isn't fixed today I'm disputing the charge with my bank.",
    "It's the Samsung 65-inch QLED. Photo attached: cracked_screen.jpg"
)


=== CUSTOMER MESSAGE ===
I just opened my new 65-inch TV from order VC-48213 and the screen is cracked. I paid $1,200 for this. If this
    isn't fixed today I'm disputing the charge with my bank.

=== STEP 1 — CLASSIFY ===
{
  "issue_type": "damaged_item",
  "urgency": "high",
  "sentiment": "angry",
  "known_info": {
    "order_number": "VC-48213",
    "item": "65-inch TV",
    "amount": "$1,200"
  },
  "summary": "Customer received a new 65-inch TV from order VC-48213 with a cracked screen. Customer paid
    $1,200 and is threatening to dispute the charge with their bank if the issue isn't fixed today."
}

=== STEP 2 — FOLLOW-UP QUESTION ===
{
  "missing_info": [
    "photo_of_damage"
  ],
  "follow_up_question": "I'm so sorry to hear about the damaged TV. To help us resolve this issue quickly for
    order VC-48213, could you please send us a photo of the cracked screen?"
}

=== CUSTOMER ANSWER (simulated) ===
It's the Samsung 65-inch QLED. Photo attached: cracked_screen.jpg

=== 

In [36]:
run_chain(
    "Hi, my headphones were supposed to arrive last Tuesday but tracking hasn't updated in a week. "
    "Order VC-51077.",
    "My zip code is 95112."
)


=== CUSTOMER MESSAGE ===
Hi, my headphones were supposed to arrive last Tuesday but tracking hasn't updated in a week. Order VC-51077.

=== STEP 1 — CLASSIFY ===
{
  "issue_type": "late_delivery",
  "urgency": "medium",
  "sentiment": "calm",
  "known_info": {
    "order_number": "VC-51077",
    "item": "headphones"
  },
  "summary": "Customer's headphones (Order VC-51077) were supposed to arrive last Tuesday and tracking hasn't
    updated in a week."
}

=== STEP 2 — FOLLOW-UP QUESTION ===
{
  "missing_info": [
    "shipping_zip_code"
  ],
  "follow_up_question": "Thanks for reaching out about your headphones, Order VC-51077. To help us investigate
    the late delivery, could you please provide the shipping zip code for your order?"
}

=== CUSTOMER ANSWER (simulated) ===
My zip code is 95112.

=== STEP 3 — PROPOSED SOLUTION ===
{
  "proposed_action": "offer_reshipment_or_refund",
  "policy_rule_used": "Late delivery: if more than 5 business days late, offer free expedited reshipment

## Iteration 1: Step 1 prompt
**v1:** "What is this customer's problem?" → returned a markdown paragraph with no fields. Step 2 had nothing structured to read.
**v2:** fixed category list, urgency definitions, "do not guess" rule, JSON-only output → Step 2 reads issue_type directly.

## Iteration 2: Step 3 prompt
**Problem:** replies claimed "we have verified the damage/your order" when no verification happened, and guessed how late the order was.
**Fix:** added rules banning verification claims and date assumptions. Reran both tests.

In [38]:
v1 = ask("What is this customer's problem? I just opened my new 65-inch TV from order VC-48213 "
         "and the screen is cracked. I paid $1,200. I'm disputing the charge with my bank.")
show("v1 OUTPUT (vague prompt)", v1)


=== v1 OUTPUT (vague prompt) ===
The customer's primary problem is that their **newly delivered 65-inch TV arrived with a cracked screen**,
    rendering it unusable.

Secondary problems and implications include:
*   They paid **$1,200 for a defective product.**
*   They are unable to use the product they purchased.
*   They are highly dissatisfied and have **already escalated the issue by disputing the charge with their
    bank**, indicating a loss of trust or belief in an easy resolution process directly with the company.
